In [25]:
#Imports and configuration
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
import langchain
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers import MultiQueryRetriever
import logging


In [30]:
# environmental variables
PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"
CHROMA_DIR = DATA_DIR / "chromadb"

# winning chunking strategy collection
COLLECTION_NAME = "recursive_100"  # (misnamed, actually recursive_1000)
# LangChain wrapper around my local Ollama model for query reformulation. Qwen because it is good at following directions
llm = ChatOllama(model="qwen3:8b")
n_queries = 5

In [13]:
# Same embedding model as used during indexing, must match exactly
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"}
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2187.17it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# LangChain wrapper around the ChromaDB collection

vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    collection_name="recursive_100",  # misnamed, actually recursive_1000
    embedding_function=embedding_model
)
print(f"Vectorstore loaded: {vectorstore._collection.count()} chunks")

Vectorstore loaded: 31741 chunks


In [33]:
# LangChain wrapper around my local Ollama model for query reformulation. Qwen because it is good at following directions
llm = ChatOllama(model="qwen3:8b")
prompt = """You are an AI assistant helping to improve information retrieval
for a scientific database about algae research.

Your task is to generate 5 alternative versions of the
given user question. Each alternative should approach the same
information need from a different angle or use different
terminology, so that together they can retrieve a broader set
of relevant documents.

Provide these alternative questions separated by newlines.
Do not number them. Do not add any explanation or preamble.
Only output the alternative questions.

Original question: {question}""" #originally 5 wasnt hardcoded, but it brought me problemas

In [ ]:
# Wraps prompt in LangChain's PromptTemplate
multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template=prompt
)

# Create the retriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),#5 is a good baseline
    llm=llm,
    prompt=multi_query_prompt,
    include_original=True  # also query with the original question
)

In [36]:
# Test ofthe retriever
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

test_query = "What is Zostera Marina and where does it grow?"

results = retriever.invoke(test_query)

print(f"\nRetrieved {len(results)} unique chunks")
for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200])

KeyError: "Input to PromptTemplate is missing variables {'n_queries'}.  Expected: ['n_queries', 'question'] Received: ['question']\nNote: if you intended {n_queries} to be part of the string and not a variable, please escape it with double curly braces like: '{{n_queries}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "